# 光迅科技(002281.SZ) 技术分析

**项目目标**: 对光迅科技进行系统性的技术分析
- 数据质量诊断
- 6大核心技术指标计算与解读
- 完整的公式推导与实现过程

**数据范围**: 2025-07-03 ~ 2026-07-03 (243个交易日)

## 1. 环境准备

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print("✓ 环境准备完成")
print(f"Pandas: {pd.__version__}, NumPy: {np.__version__}")

## 2. 数据加载与诊断

In [ ]:
# 加载数据
df = pd.read_csv('../data/002281_daily.csv', encoding='utf-8-sig')

# 日期转换
df['trade_date'] = pd.to_datetime(df['trade_date'])
df = df.sort_values('trade_date').reset_index(drop=True)

print(f"✓ 数据加载完成: {len(df)} 行")

### 2.1 基础信息检查

In [ ]:
# 数据维度
print("📊 数据维度:")
print(f"   行数: {df.shape[0]}, 列数: {df.shape[1]}")
print()

# 数据类型
print(" 数据类型:")
print(df.dtypes)
print()

# 前5行
print(" 前5行数据:")
print(df.head())

### 2.2 缺失值检查

In [ ]:
# 缺失值统计
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

print("🔍 缺失值分析:")
print(missing[missing > 0] if missing.sum() > 0 else "✓ 无缺失值")
print()

# 重复值
duplicates = df.duplicated().sum()
print(f"🔍 重复值: {duplicates} 行")

### 2.3 描述性统计

In [ ]:
# 数值列的统计量
print("📈 描述性统计:")
print(df[['open', 'high', 'low', 'close', 'vol', 'pct_chg']].describe().round(2))
print()

# 关键指标
print("🎯 关键指标:")
print(f"   最高价:    {df['high'].max():.2f}  ({df.loc[df['high'].idxmax(), 'trade_date'].date()})")
print(f"   最低价:    {df['low'].min():.2f}  ({df.loc[df['low'].idxmin(), 'trade_date'].date()})")
print(f"   平均成交量: {df['vol'].mean()/1e4:.1f} 万手")
print(f"   平均涨跌幅: {df['pct_chg'].mean():.2f}%")

## 3. 技术指标详解与计算

### 3.1 RSI - 相对强弱指标

**公式**:  

RSI = 100 - 100 / (1 + RS)

其中:  
- RS = 平均上涨幅度 / 平均下跌幅度
- 平均涨幅 = EMA(Gain, N), N=14
- 平均跌幅 = EMA(Loss, N), N=14

**解读**:
- RSI > 70: 超买区, 可能回调
- RSI < 30: 超卖区, 可能反弹
- RSI = 50: 多空均衡

In [ ]:
# RSI 计算函数
def calc_rsi(close, period=14):
    """
    计算 RSI
    参数: close - 收盘价序列, period - 周期(默认14)
    返回: RSI 序列
    """
    delta = close.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)
    
    avg_gain = gain.ewm(com=period-1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period-1, min_periods=period).mean()
    
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# 计算 RSI
df['RSI'] = calc_rsi(df['close'], period=14)

print("✓ RSI 计算完成")
print(f"\n最新 RSI(14): {df['RSI'].iloc[-1]:.2f}")

### 3.2 MACD - 指数平滑异同移动平均线

**公式**:  

- EMA_fast = EMA(Close, 12) — 快速EMA
- EMA_slow = EMA(Close, 26) — 慢速EMA
- DIF = EMA_fast - EMA_slow — 离差值
- DEA = EMA(DIF, 9) — 信号线
- MACD柱 = 2 × (DIF - DEA)

**解读**:
- DIF > DEA: 金叉, 看多
- DIF < DEA: 死叉, 看空
- MACD柱 > 0: 多头动能
- MACD柱 < 0: 空头动能

In [ ]:
# MACD 计算函数
def calc_macd(close, fast=12, slow=26, signal=9):
    """
    计算 MACD
    返回: (DIF, DEA, MACD柱)
    """
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()
    
    dif = ema_fast - ema_slow
    dea = dif.ewm(span=signal, adjust=False).mean()
    macd_hist = 2 * (dif - dea)
    
    return dif, dea, macd_hist

# 计算 MACD
df['DIF'], df['DEA'], df['MACD'] = calc_macd(df['close'])

print("✓ MACD 计算完成")
print(f"\n最新值 (2026-07-03):")
print(f"   DIF:   {df['DIF'].iloc[-1]:.4f}")
print(f"   DEA:   {df['DEA'].iloc[-1]:.4f}")
print(f"   MACD:  {df['MACD'].iloc[-1]:.4f}")
print(f"   金叉/死叉: {'金叉' if df['DIF'].iloc[-1] > df['DEA'].iloc[-1] else '死叉'}")

### 3.3 布林带 (Bollinger Bands)

**公式**:  

- 中轨 = SMA(Close, 20) — 20日均线
- 标准差 = StdDev(Close, 20)
- 上轨 = 中轨 + 2 × 标准差
- 下轨 = 中轨 - 2 × 标准差

**解读**:
- 价格 > 上轨: 超买, 可能回调
- 价格 < 下轨: 超卖, 可能反弹
- 带宽收窄: 波动率降低, 变盘信号
- 价格在中轨上方: 偏多运行

In [ ]:
# 布林带计算函数
def calc_bollinger(close, period=20, num_std=2):
    """
    计算布林带
    返回: (上轨, 中轨, 下轨)
    """
    mid = close.rolling(window=period).mean()
    std = close.rolling(window=period).std()
    
    upper = mid + num_std * std
    lower = mid - num_std * std
    
    return upper, mid, lower

# 计算布林带
df['BOLL_UP'], df['BOLL_MID'], df['BOLL_LOW'] = calc_bollinger(df['close'], 20, 2)

print("✓ 布林带计算完成")
print(f"\n最新值 (2026-07-03):")
print(f"   上轨: {df['BOLL_UP'].iloc[-1]:.2f}")
print(f"   中轨: {df['BOLL_MID'].iloc[-1]:.2f}")
print(f"   下轨: {df['BOLL_LOW'].iloc[-1]:.2f}")
print(f"   带宽: {(df['BOLL_UP'].iloc[-1] - df['BOLL_LOW'].iloc[-1]):.2f}")

### 3.4 WR - 威廉指标

**公式**:  

WR = (N日最高价 - 收盘价) / (N日最高价 - N日最低价) × (-100)

**解读**:
- WR > -20: 超买区
- WR < -80: 超卖区
- 与 RSI 类似, 但方向相反 (RSI 越大越强, WR 越大越弱)

In [ ]:
# WR 计算函数
def calc_wr(high, low, close, period=14):
    """
    计算威廉指标
    返回: WR 序列
    """
    high_n = high.rolling(window=period).max()
    low_n = low.rolling(window=period).min()
    
    wr = (high_n - close) / (high_n - low_n) * (-100)
    return wr

# 计算 WR
df['WR'] = calc_wr(df['high'], df['low'], df['close'], period=14)

print("✓ WR 计算完成")
print(f"\n最新 WR(14): {df['WR'].iloc[-1]:.2f}")

### 3.5 ATR - 真实波幅

**公式**:  

- TR = max(H-L, |H-PrevClose|, |L-PrevClose|) — 真实波幅
- ATR = MA(TR, N), N=14

**解读**:
- ATR 越大: 日波动越大, 风险越高
- 止损参考: 1.5~2 × ATR
- ATR 上升: 趋势加速
- ATR 下降: 行情收敛

In [ ]:
# ATR 计算函数
def calc_atr(high, low, close, period=14):
    """
    计算 ATR
    返回: ATR 序列
    """
    prev_close = close.shift(1)
    
    tr1 = high - low
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()
    
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(window=period).mean()
    
    return atr

# 计算 ATR
df['ATR'] = calc_atr(df['high'], df['low'], df['close'], period=14)

print("✓ ATR 计算完成")
print(f"\n最新 ATR(14): {df['ATR'].iloc[-1]:.2f}")
print(f"ATR 占股价比例: {df['ATR'].iloc[-1]/df['close'].iloc[-1]*100:.2f}%")

### 3.6 CR - 能量指标

**公式**:  

CR = 100 × Σ(高价 - 中间价) / Σ(中间价 - 低价)

其中: 中间价 = (前日最高 + 前日最低 + 前日收盘) / 3

**解读**:
- CR > 100: 多方占优
- CR < 100: 空方占优
- CR > 200/300: 超强, 警惕回调
- CR < 40: 超卖, 关注反弹

**参考线**: 40, 60, 160, 200

In [ ]:
# CR 计算函数
def calc_cr(high, low, close, period=26):
    """
    计算 CR 能量指标
    返回: CR 序列
    """
    prev_high = high.shift(1)
    prev_low = low.shift(1)
    prev_close = close.shift(1)
    
    # 中间价
    pm = (prev_high + prev_low + prev_close) / 3
    
    # 多头力量 & 空头力量
    bull_power = (high - pm).clip(lower=0)
    bear_power = (pm - low).clip(lower=0)
    
    # CR 计算
    cr = 100 * bull_power.rolling(window=period).sum() / bear_power.rolling(window=period).sum()
    
    return cr

# 计算 CR
df['CR'] = calc_cr(df['high'], df['low'], df['close'], period=26)

print("✓ CR 计算完成")
print(f"\n最新 CR(26): {df['CR'].iloc[-1]:.2f}")

## 4. 可视化分析

In [ ]:
# 创建图表
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('光迅科技(002281) 技术指标可视化', fontsize=16, fontweight='bold')

# 子图1: 价格 + 布林带
ax1 = axes[0, 0]
ax1.plot(df['trade_date'], df['close'], label='Close', color='black', linewidth=1.5)
ax1.plot(df['trade_date'], df['BOLL_UP'], label='Upper', color='red', linestyle='--', alpha=0.7)
ax1.plot(df['trade_date'], df['BOLL_MID'], label='Mid', color='blue', linestyle='--', alpha=0.7)
ax1.plot(df['trade_date'], df['BOLL_LOW'], label='Lower', color='green', linestyle='--', alpha=0.7)
ax1.fill_between(df['trade_date'], df['BOLL_UP'], df['BOLL_LOW'], alpha=0.1)
ax1.set_title('布林带 (Bollinger Bands)', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 子图2: MACD
ax2 = axes[0, 1]
ax2.plot(df['trade_date'], df['DIF'], label='DIF', color='red')
ax2.plot(df['trade_date'], df['DEA'], label='DEA', color='blue')
colors = ['red' if v > 0 else 'green' for v in df['MACD']]
ax2.bar(df['trade_date'], df['MACD'], alpha=0.6, color=colors)
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.set_title('MACD 指标', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

# 子图3: RSI
ax3 = axes[1, 0]
ax3.plot(df['trade_date'], df['RSI'], label='RSI(14)', color='purple')
ax3.axhline(y=70, color='red', linestyle='--', alpha=0.5, label='超买(70)')
ax3.axhline(y=30, color='green', linestyle='--', alpha=0.5, label='超卖(30)')
ax3.axhline(y=50, color='gray', linestyle='--', alpha=0.3, label='均衡(50)')
ax3.set_ylim(0, 100)
ax3.set_title('RSI 相对强弱指标', fontsize=12)
ax3.legend()
ax3.grid(True, alpha=0.3)

# 子图4: WR
ax4 = axes[1, 1]
ax4.plot(df['trade_date'], df['WR'], label='WR(14)', color='orange')
ax4.axhline(y=-20, color='red', linestyle='--', alpha=0.5, label='超买(-20)')
ax4.axhline(y=-80, color='green', linestyle='--', alpha=0.5, label='超卖(-80)')
ax4.set_ylim(-100, 0)
ax4.set_title('威廉指标 (WR)', fontsize=12)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 第二个图表: ATR + CR
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ATR
ax1 = axes[0]
ax1.plot(df['trade_date'], df['ATR'], color='brown', label='ATR(14)')
ax1.fill_between(df['trade_date'], df['ATR'], alpha=0.3)
ax1.set_title('ATR 真实波幅', fontsize=12)
ax1.set_ylabel('ATR值')
ax1.legend()
ax1.grid(True, alpha=0.3)

# CR
ax2 = axes[1]
ax2.plot(df['trade_date'], df['CR'], color='darkblue', label='CR(26)')
ax2.axhline(y=100, color='gray', linestyle='--', alpha=0.5, label='均衡线(100)')
ax2.axhline(y=200, color='red', linestyle=':', alpha=0.7, label='超强(200)')
ax2.axhline(y=40, color='green', linestyle=':', alpha=0.7, label='超卖(40)')
ax2.fill_between(df['trade_date'], df['CR'], 100, where=(df['CR']>100), alpha=0.2, color='red')
ax2.fill_between(df['trade_date'], df['CR'], 100, where=(df['CR']<100), alpha=0.2, color='green')
ax2.set_ylim(0, 300)
ax2.set_title('CR 能量指标', fontsize=12)
ax2.set_ylabel('CR值')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 综合研判

### 5.1 当前指标状态 (2026-07-03)

In [ ]:
latest = df.iloc[-1]

print("=" * 60)
print("光迅科技(002281) 技术面综合研判")
print("=" * 60)
print(f"日期: {latest['trade_date'].date()}")
print()

print("【价格位置】")
print(f"  收盘价: {latest['close']:.2f}")
print(f"  布林中轨: {latest['BOLL_MID']:.2f} ({'上方' if latest['close'] > latest['BOLL_MID'] else '下方'})")
print()

print("【动量指标】")
print(f"  RSI(14): {latest['RSI']:.2f} ({'超买' if latest['RSI'] > 70 else '超卖' if latest['RSI'] < 30 else '中性'})")
print(f"  WR(14): {latest['WR']:.2f} ({'超买' if latest['WR'] > -20 else '超卖' if latest['WR'] < -80 else '中性'})")
print()

print("【趋势指标】")
print(f"  MACD DIF: {latest['DIF']:.4f}")
print(f"  MACD DEA: {latest['DEA']:.4f}")
print(f"  MACD 柱: {latest['MACD']:.4f}")
print(f"  信号: {'金叉(看多)' if latest['DIF'] > latest['DEA'] else '死叉(看空)'}")
print()

print("【波动率】")
print(f"  ATR(14): {latest['ATR']:.2f} ({latest['ATR']/latest['close']*100:.2f}%)")
print(f"  布林带宽: {latest['BOLL_UP'] - latest['BOLL_LOW']:.2f}")
print()

print("【成交量】")
print(f"  CR(26): {latest['CR']:.2f} ({'多方占优' if latest['CR'] > 100 else '空方占优'})")
print()

# 综合建议
print("=" * 60)
print("【综合研判】")
print("=" * 60)

if latest['RSI'] < 30 and latest['WR'] < -80:
    print("✅ 超卖区域, 短期可能反弹")
elif latest['RSI'] > 70 and latest['WR'] > -20:
    print("⚠️ 超买区域, 警惕回调风险")
elif latest['DIF'] > latest['DEA']:
    print("🟢 多头趋势维持")
else:
    print("🔴 空头趋势, 观望为主")

### 5.2 风险提示
- ⚠️ 技术指标仅作为参考, 不构成投资建议
- 📌 需结合基本面、市场情绪综合判断
- 💡 建议设置严格止损, 控制仓位风险